# 2026.04.02 작업 상황

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv('OPENAI_API_KEY')
print(f"✅ API Key 로드: {'성공' if api_key else '실패 → .env 파일 확인 필요'}")

✅ API Key 로드: 성공


In [8]:
# RAG_DOCS 
RAG_DOCS = [
    "Choices_US_HB.pdf",        
    "Overseas_HB.pdf",
    "Pharmacy_HB.pdf",
    "NGR_HB.pdf",
    "TFL_HB.pdf",
    "TRICARE_ADDP_HB.pdf",
    "ADDP_Brochure.pdf",
    "Maternity_Br.pdf",
    "Medicare_Turning_65_Br.pdf",
    "Medicare_Under_65_Br.pdf",
    "Retiring_AD_Br.pdf",
    "Retiring_NGR_Br.pdf",
    "QLEs_FS.pdf",
]

# TABLE_DOCS 
TABLE_DOCS = [
    "Costs_Fees.pdf",
    "Pharmacy_HB.pdf",
    "NGR_HB.pdf",
    "TFL_HB.pdf",
    "Plans_Overview_FS.pdf",
    # Choices_US_HB는 비용 표가 Costs_Fees에 이미 있으므로 제외
]

In [9]:
# 파일 경로 (객체명) - 6개
# 1. Pharmacy_HB: 약국 프로그램 안내서
# 2. NGR_HB: 국가방위군 및 예비군을 위한 트라이케어 안내서
# 3. TFL_HB : 평생 트라이케어
# 4. ADDP_Brochure : 현역 군인 치과 프로그램 안내서
# 5. PDF_Overseas_HB : 해외 프로그램 안내서

# ADDP_Brochure= 'C:\\SKN-_24\\제 3차 프로젝트\data\\TRICARE_ADDP_HB_FINAL_508c(현역 군인 치과 프로그램 안내서).pdf'    
# Pharmacy_HB= 'C:\\SKN-_24\\제 3차 프로젝트\\data\\Pharmacy_HB(tricare 약국 프로그램 안내서).pdf'
# Overseas_HB= 'C:\\SKN-_24\\제 3차 프로젝트\\data\\Overseas_HB(해외 프로그램 안내서).pdf'
NGR_HB= 'C:\\SKN-_24\\제 3차 프로젝트\\data\\NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf'
TFL_HB= 'C:\\SKN-_24\제 3차 프로젝트\\data\\TFL_HB(평생 트라이케어).pdf'

for path in [NGR_HB, TFL_HB]:
    status = '✅' if os.path.exists(path) else '❌'
    print(f'{status} {path}')   # 잘 가져왔는지 확인

✅ C:\SKN-_24\제 3차 프로젝트\data\NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf
✅ C:\SKN-_24\제 3차 프로젝트\data\TFL_HB(평생 트라이케어).pdf


In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# GPU가 없는 환경이면 'device': 'cpu'로 변경
embedding_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    #model_kwargs={'device': 'cuda'}  # GPU 환경 -> GPU 없어서 주석 처리
    model_kwargs={'device': 'cpu'}  # CPU 환경
)

print('✅ BAAI/bge-m3 임베딩 모델 로드 완료')

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ BAAI/bge-m3 임베딩 모델 로드 완료


In [12]:
from langchain_community.document_loaders import PyMuPDFLoader

# NGR_HB / TFL_HB
loader_handbook= PyMuPDFLoader(NGR_HB)
loader_handbook= PyMuPDFLoader(TFL_HB)
loader_costs= PyMuPDFLoader(TFL_HB)
loader_costs= PyMuPDFLoader(NGR_HB)

docs_handbook= loader_handbook.load()
docs_costs= loader_costs.load()

docs_handbook= loader_handbook.load()

# 두 PDF를 하나의 리스트로 합침
all_pdf_docs1 = docs_costs + docs_handbook

In [17]:
print(f'1. Costs_Fees.pdf: {len(docs_costs)}페이지')
print(f'2. TOP_Handbook.pdf: {len(docs_handbook)}페이지')
print(f'=> 합계: {len(all_pdf_docs1)}페이지')
print(f'\n--- 샘플 메타데이터 ---')
print(all_pdf_docs1[0].metadata)
print(all_pdf_docs1[3].metadata)   # Costs_Fees.pdf 마지막 페이지 (총 4페이지)
print(all_pdf_docs1[18].metadata)   # TOP_Handbook.pdf 1페이지 (docs_costs 4개 다음부터)

1. Costs_Fees.pdf: 16페이지
2. TOP_Handbook.pdf: 44페이지
=> 합계: 60페이지

--- 샘플 메타데이터 ---
{'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.5 (Macintosh)', 'creationdate': '2025-12-22T11:40:54-05:00', 'source': 'C:\\SKN-_24\\제 3차 프로젝트\\data\\NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf', 'file_path': 'C:\\SKN-_24\\제 3차 프로젝트\\data\\NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf', 'total_pages': 16, 'format': 'PDF 1.6', 'title': 'TRICARE Choices for National Guard and Reserve Handbook', 'author': 'TRICARE Communications', 'subject': '', 'keywords': '', 'moddate': '2026-01-09T11:34:36-05:00', 'trapped': '', 'modDate': "D:20260109113436-05'00'", 'creationDate': "D:20251222114054-05'00'", 'page': 0}
{'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.5 (Macintosh)', 'creationdate': '2025-12-22T11:40:54-05:00', 'source': 'C:\\SKN-_24\\제 3차 프로젝트\\data\\NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf', 'file_path': 'C:\\SKN-_24\\제 3차 프로젝트\\data\\NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf', 'tota

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, # 청크 하나의 최대 글자 수(500자 넘으면 자름)
    chunk_overlap=50,   #(앞 청크의 마지막 50자를 다음 청크 앞에서 반복 포함 -> 청크 경계에서 문맥이 끊기는 것 방지)
    separators=['\n\n', '\n', '. ', ' ', '']    #자를 때 우선 순위로 자름. (\n\n: 문단 자르기, \n: 줄바꿈, .: 문장, 그 다음 공백, 마지막 글자 단위)
)

pdf_chunks = text_splitter.split_documents(all_pdf_docs1)


In [20]:
print(f' 분할 결과: {len(all_pdf_docs1)}페이지 → {len(pdf_chunks)}개 청크')
print(f'\n--- 청크 샘플 ---')
print(pdf_chunks[0].page_content[:300])
print(f'\nMetadata: {pdf_chunks[0].metadata}')  #PDF 로더가 알아서 채움

 분할 결과: 60페이지 → 324개 청크

--- 청크 샘플 ---
TRICARE®Choices for 
National Guard and Reserve 
HANDBOOK 
TRICARE offers comprehensive, affordable health care, 
pharmacy, and dental coverage to meet your changing needs.

Metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.5 (Macintosh)', 'creationdate': '2025-12-22T11:40:54-05:00', 'source': 'C:\\SKN-_24\\제 3차 프로젝트\\data\\NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf', 'file_path': 'C:\\SKN-_24\\제 3차 프로젝트\\data\\NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf', 'total_pages': 16, 'format': 'PDF 1.6', 'title': 'TRICARE Choices for National Guard and Reserve Handbook', 'author': 'TRICARE Communications', 'subject': '', 'keywords': '', 'moddate': '2026-01-09T11:34:36-05:00', 'trapped': '', 'modDate': "D:20260109113436-05'00'", 'creationDate': "D:20251222114054-05'00'", 'page': 0}


## ✅ 중간 점검
### PDF 청킹 — RecursiveCharacterTextSplitter

### chunk_size=500, chunk_overlap=50 까지 수행

---

In [ ]:
ADDP_Brochure= 'C:\\SKN-_24\\제 3차 프로젝트\data\\TRICARE_ADDP_HB_FINAL_508c(현역 군인 치과 프로그램 안내서).pdf'    
Pharmacy_HB= 'C:\\SKN-_24\\제 3차 프로젝트\\data\\Pharmacy_HB(tricare 약국 프로그램 안내서).pdf'
Overseas_HB= 'C:\\SKN-_24\\제 3차 프로젝트\\data\\Overseas_HB(해외 프로그램 안내서).pdf'